# Validación LDA — Solo Cercanías (04)

**Parámetros:** `n_topics=16`, `alpha=0.5`, `beta=0.01`, `filter_prefixes=['04']`, `use_periods=True` (7 franjas), `token_granularity='temporal'`

Pruebas clave:
1. Ajuste e integridad de theta
2. Heatmap theta (estaciones × tópicos)
3. Perfil IN/OUT por franja de cada tópico
4. Estación más representativa por tópico
5. **Vecinos más cercanos en espacio de tópicos**

In [ ]:
import sys, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, '.')
from region_function_lda import RegionFunctionLDA

MATRICES_DIR = Path('../../cubo_izquierdo/matrices')
OD_FOLDERS = [
    'validaciones_3_7_febrero',
    'validaciones_8_9_febrero',
    'validaciones_10_16_febrero',
    'validaciones_17_23_febrero',
    'validaciones_24febrero_2marzo',
]

frames = []
for folder in OD_FOLDERS:
    p = MATRICES_DIR / folder / 'matriz_od.csv'
    df = pd.read_csv(p)
    frames.append(df)
    print(f'{folder}: {len(df):,} filas')

od = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(od):,} filas')

# Mapa punto_acceso → nombre para etiquetas
nombres = dict(zip(od['origen_punto_acceso'], od['origen_nombre']))
nombres.update(dict(zip(od['destino_punto_acceso'], od['destino_nombre'])))

def label(sid, max_len=30):
    n = nombres.get(sid, '')
    return f'{n[:max_len]}  ({sid})' if n else sid

---
## 1 — Ajuste del modelo

In [ ]:
# alpha=0.5: concentración moderada sobre 16 tópicos (ni uniforme ni espiga pura).
# beta=0.01: prior muy esparso sobre los 28 tokens (2 dir × 7 franjas × 2 daytypes)
#   → cada tópico se concentrará en 1-2 combinaciones (dirección, franja) concretas,
#     lo que produce tópicos muy interpretables funcionalmente.

model = RegionFunctionLDA(
    n_topics=16,
    slot_minutes=15,
    alpha=0.5,
    beta=0.01,
    max_iter=200,
    filter_prefixes=['04'],
    use_periods=True,
    min_df=1,
    log_counts=False,
    token_granularity='temporal',
    min_station_trips=0,
    n_init=3,
    random_state=42,
)

t0 = time.time()
theta = model.fit_transform(od)
elapsed = time.time() - t0

print(f'Ajuste completado: {elapsed:.1f}s')
print(f'Theta shape:       {theta.shape}  (estaciones × tópicos)')
print(f'Estaciones:        {len(theta)}')
theta.head()

---
## 2 — Integridad de theta

In [ ]:
sumas = theta.sum(axis=1)
ok_suma  = ((sumas - 1.0).abs() < 1e-4).all()
ok_rango = (theta.values.min() >= 0) and (theta.values.max() <= 1)

print(f'Filas suman 1:        {"✓" if ok_suma  else "✗"}  (min={sumas.min():.6f}, max={sumas.max():.6f})')
print(f'Valores en [0, 1]:    {"✓" if ok_rango else "✗"}  (min={theta.values.min():.6f}, max={theta.values.max():.6f})')
print(f'Sin NaN:              {"✓" if not theta.isnull().any().any() else "✗"}')

# Diversidad de tópico dominante
topic_cols = [f'topic_{i}' for i in range(model.n_topics)]
dom = theta.idxmax(axis=1).value_counts().reindex(topic_cols, fill_value=0)
pct_max = dom.max() / len(theta) * 100
print(f'\nEstaciones por tópico dominante (max concentración: {pct_max:.0f}%):')
print(dom.to_string())

---
## 3 — Heatmap theta (estaciones × tópicos)

In [ ]:
# Fix 1: operar solo sobre columnas numéricas (evita TypeError al mezclar str+float)
# Fix 2: sort numérico en lugar de lexicográfico (topic_10 después de topic_9, no de topic_1)

topic_cols = [f'topic_{i}' for i in range(model.n_topics)]
th = theta[topic_cols].copy()

dom_col  = th.idxmax(axis=1)                           # 'topic_3', 'topic_12'...
dom_num  = dom_col.str.extract(r'(\d+)').astype(int)   # entero para sort correcto
dom_peso = th.max(axis=1)

order_idx = (
    pd.DataFrame({'_num': dom_num.values.ravel(), '_peso': dom_peso.values}, index=th.index)
    .sort_values(['_num', '_peso'], ascending=[True, False])
    .index
)
th = th.loc[order_idx]

n = len(th)
fig, ax = plt.subplots(figsize=(11, max(6, n * 0.25)))
im = ax.imshow(th.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=th.values.max())

ax.set_title(
    f'Theta — Cercanías (04)\n'
    f'16 tópicos · α=0.5 · β=0.01  ({n} estaciones, ordenadas por tópico dominante)',
    fontsize=10
)
ax.set_xlabel('Tópico')
ax.set_xticks(range(model.n_topics))
ax.set_xticklabels([f'T{i}' for i in range(model.n_topics)], fontsize=8)
ax.set_yticks(range(n))
ax.set_yticklabels(
    [f'{nombres.get(s, s)[:30]}  ({s})' for s in th.index],
    fontsize=6
)
plt.colorbar(im, ax=ax, label='Peso del tópico', shrink=0.5)

# Líneas blancas separadoras entre grupos de tópico dominante
prev = dom_num.loc[order_idx].values.ravel()[0]
for k, v in enumerate(dom_num.loc[order_idx].values.ravel()):
    if v != prev:
        ax.axhline(k - 0.5, color='white', linewidth=1.2, alpha=0.8)
        prev = v

plt.tight_layout()
plt.show()

In [ ]:
# th ya está bien ordenado desde la celda anterior (solo columnas numéricas)
X = th.values.astype(float)
sim_matrix = cosine_similarity(X)

n = len(th)
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_title('Similitud coseno entre estaciones en espacio LDA\n(ordenadas por tópico dominante)', fontsize=10)
ax.set_xticks(range(n))
ax.set_xticklabels([s for s in th.index], rotation=90, fontsize=4)
ax.set_yticks(range(n))
ax.set_yticklabels([f'{nombres.get(s, s)[:22]}' for s in th.index], fontsize=4)
plt.colorbar(im, ax=ax, label='Similitud coseno', shrink=0.5)

# Separadores de grupo
prev = dom_num.loc[order_idx].values.ravel()[0]
for k, v in enumerate(dom_num.loc[order_idx].values.ravel()):
    if v != prev:
        ax.axhline(k - 0.5, color='white', linewidth=0.8)
        ax.axvline(k - 0.5, color='white', linewidth=0.8)
        prev = v

plt.tight_layout()
plt.show()

---
## 4 — Perfil IN/OUT por franja de cada tópico

In [ ]:
BIN_ORDER = ['noche', 'madrug', 'punta_man', 'mediodia', 'tarde', 'punta_tar', 'noche_ini']

def parse_token(token):
    """Extrae (dirección, franja) de un token 'OUT_punta_man_WD'."""
    direccion = 'OUT' if token.startswith('OUT') else 'IN'
    inner = token[len(direccion) + 1:]   # 'punta_man_WD'
    franja = inner.rsplit('_', 1)[0]     # 'punta_man'
    return direccion, franja

topics_df = model.inspect_topics(n_top_tokens=28)
n_topics  = model.n_topics
cols_grid = 4
rows_grid = (n_topics + cols_grid - 1) // cols_grid

fig, axes = plt.subplots(rows_grid, cols_grid, figsize=(cols_grid * 4, rows_grid * 3))
axes = axes.flatten()

for i, (topic, row) in enumerate(topics_df.iterrows()):
    bins_out, bins_in = {}, {}
    for token, peso in zip(row['top_tokens'], row['top_weights']):
        dir_, franja = parse_token(token)
        if dir_ == 'OUT':
            bins_out[franja] = bins_out.get(franja, 0) + peso
        else:
            bins_in[franja]  = bins_in.get(franja, 0) + peso

    ax = axes[i]
    all_keys = [b for b in BIN_ORDER if b in bins_out or b in bins_in]
    x_pos = range(len(all_keys))
    out_v = [bins_out.get(b, 0) for b in all_keys]
    in_v  = [bins_in.get(b, 0)  for b in all_keys]
    w = 0.38
    ax.bar([x - w/2 for x in x_pos], out_v, width=w, color='steelblue', alpha=0.85, label='OUT')
    ax.bar([x + w/2 for x in x_pos], in_v,  width=w, color='darkorange', alpha=0.85, label='IN')
    ax.set_xticks(list(x_pos))
    ax.set_xticklabels(all_keys, rotation=45, ha='right', fontsize=6)
    ax.set_title(topic, fontsize=9)
    if i == 0:
        ax.legend(fontsize=6)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribución horaria por tópico — Cercanías (azul=OUT, naranja=IN)\nβ=0.01: cada tópico se concentra en 1-2 franjas', fontsize=11)
plt.tight_layout()
plt.show()

---
## 5 — Estación más representativa por tópico

In [ ]:
print('Estación más representativa de cada tópico (cercanías):\n')
for col in theta.columns:
    best = theta[col].idxmax()
    val  = theta[col].max()
    nom  = nombres.get(best, '')
    print(f'  {col}: {best}  {nom:<40}  ({val:.3f})')

---
## 6 — Vecinos más cercanos en espacio de tópicos

In [ ]:
# Estaciones clave para validación cualitativa
# Representan perfiles muy distintos: hub central, terminal sierra, universitaria,
# corredor obrero, periferia residencial
KEY_STATIONS = {
    '04_L0_P108': 'SOL (hub central)',
    '04_L0_P7':   'ATOCHA (gran hub)',
    '04_L0_P13':  'CHAMARTÍN (hub norte)',
    '04_L0_P62':  'PRÍNCIPE PÍO',
    '04_L0_P10':  'CANTOBLANCO UNIV.',
    '04_L0_P11':  'CERCEDILLA (sierra)',
    '04_L0_P57':  'PARLA (sur)',
    '04_L0_P1':   'ALCALÁ DE HENARES',
    '04_L0_P116': 'ROBLEDO DE CHAVELA',
    '04_L0_P77':  'TRES CANTOS',
}
K = 7

X = theta.values.astype(float)
sim_full = cosine_similarity(X)
station_list = list(theta.index)

print(f'Top-{K} vecinos en espacio LDA (similitud coseno sobre theta)\n')
for sid, desc in KEY_STATIONS.items():
    if sid not in station_list:
        print(f'  {desc} ({sid}): NO encontrada en theta\n')
        continue
    idx  = station_list.index(sid)
    sims = sim_full[idx]
    top  = np.argsort(sims)[::-1][1:K+1]
    print(f'{desc}:')
    for rank, j in enumerate(top, 1):
        print(f'  {rank}. {nombres.get(station_list[j], station_list[j]):<35} ({station_list[j]})  sim={sims[j]:.4f}')
    print()

In [ ]:
# Visualización: perfiles de theta para cada estación + sus K vecinos
K_VIS = 4   # vecinos a mostrar por gráfico

for sid, desc in KEY_STATIONS.items():
    if sid not in station_list:
        continue
    idx  = station_list.index(sid)
    sims = sim_full[idx]
    top  = np.argsort(sims)[::-1][1:K_VIS+1]
    group = [sid] + [station_list[j] for j in top]

    fig, axes = plt.subplots(1, len(group), figsize=(len(group) * 3.5, 3.2), sharey=True)
    x = range(model.n_topics)

    for k, (ax, s) in enumerate(zip(axes, group)):
        perfil = theta.loc[s].values
        color  = 'steelblue' if k == 0 else 'tomato'
        alpha  = 1.0 if k == 0 else 0.75
        ax.bar(x, perfil, color=color, alpha=alpha)
        ax.set_ylim(0, max(theta.values.max() * 1.05, 0.05))
        ax.set_xticks(list(x))
        ax.set_xticklabels([f'T{i}' for i in x], rotation=90, fontsize=6)
        nom = nombres.get(s, s)
        sim_v = sims[station_list.index(s)]
        ttl = f'{nom[:22]}\n({s})'
        if k > 0:
            ttl += f'\nsim={sim_v:.3f}'
        ax.set_title(ttl, fontsize=7)

    fig.suptitle(f'{desc} — vecinos más cercanos en theta', fontsize=9)
    plt.tight_layout()
    plt.show()

---
## 7 — PCA de theta (coloreado por tópico dominante)

In [ ]:
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from IPython.display import HTML

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(theta.values.astype(float))

dom_topic = theta.idxmax(axis=1).values
unique_topics = sorted(set(dom_topic), key=lambda t: int(t.split('_')[1]))

fig = go.Figure()
for t in unique_topics:
    mask = dom_topic == t
    labels_hover = [f'{nombres.get(s, s)}<br>{s}' for s, m in zip(theta.index, mask) if m]
    fig.add_trace(go.Scatter(
        x=coords[mask, 0],
        y=coords[mask, 1],
        mode='markers',
        name=t,
        text=labels_hover,
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra></extra>',
        marker=dict(size=9, opacity=0.85),
    ))

fig.update_layout(
    title=f'PCA sobre theta — Cercanías (04)<br>'
          f'<sup>PC1={pca.explained_variance_ratio_[0]*100:.1f}%  '
          f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%  |  '
          f'color = tópico dominante</sup>',
    xaxis_title='PC1',
    yaxis_title='PC2',
    height=600,
    hovermode='closest',
)
HTML(fig.to_html(include_plotlyjs='cdn', full_html=False))

---
## 8 — Guardar theta

In [ ]:
out_dir = Path('output')
out_dir.mkdir(exist_ok=True)

theta.to_parquet(out_dir / 'theta_cercanias.parquet')
theta.round(6).to_csv(out_dir / 'theta_cercanias.csv')
model.inspect_topics(15).to_csv(out_dir / 'topics_cercanias.csv')

print(f'Guardado en: {out_dir.resolve()}')
print(f'  theta_cercanias.parquet  {theta.shape}')
print(f'  topics_cercanias.csv')